# Fine-tune NandiBERT on Naamapadam: Indic named-entity recognition

Naamapadam is an NER dataset for 11 Indic languages. Every sentence is a list of words, and every word has one tag: `O` (not an entity), or `B-`/`I-` (beginning/inside) of a person (`PER`), organisation (`ORG`) or location (`LOC`).

This is **token classification**: the model predicts a label for every token, not one label for the whole input.

## 1. Install and import

Run this once in a fresh environment. `seqeval` computes entity-level precision, recall and F1, the standard NER metrics.

In [ ]:
!pip install -U "transformers>=5.0" "datasets<4" accelerate seqeval

In [1]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    DataCollatorForTokenClassification, Trainer, TrainingArguments
)
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA: True
NVIDIA GeForce RTX 4090


## 2. Load the data

The dataset repo stores a loading script and zip files on its `main` branch. The Hub keeps a parquet copy of it on the `refs/convert/parquet` branch, which we load directly (`%2F` is the `/` in the branch name).

Languages: `as`, `bn`, `gu`, `hi`, `kn`, `ml`, `mr`, `or`, `pa`, `ta`, `te`.

In [2]:
LANG = "hi"

base = f"hf://datasets/ai4bharat/naamapadam@refs%2Fconvert%2Fparquet/{LANG}"
data_files = {
    "train": f"{base}/train/*.parquet",
    "validation": f"{base}/validation/*.parquet",
    "test": f"{base}/test/*.parquet",
}
dataset = load_dataset("parquet", data_files=data_files)

# A few sentences have no words at all; drop them.
dataset = dataset.filter(lambda x: len(x["tokens"]) > 0)

print(dataset)
print(dataset["train"][0])

Filter:   0%|          | 0/985787 [00:00<?, ? examples/s]

Filter:   0%|          | 0/13460 [00:00<?, ? examples/s]

Filter:   0%|          | 0/867 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 985787
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 13460
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 864
    })
})
{'tokens': ['सेक्टर', '55/56', 'के', 'एसएचओ', 'अरविंद', 'कुमार', 'ने', 'बताया', 'कि', 'इस', 'मामले', 'में', 'आईपीसी', 'की', 'धारा', '376', '-', 'डी', '(', 'गैंगरेप', ')', 'के', 'तहत', 'मामला', 'दर्ज', 'कर', 'लिया', 'गया', 'है', '।'], 'ner_tags': [0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


The tag names are stored in the dataset itself, so we read them from there instead of typing them out.

In [3]:
label_names = dataset["train"].features["ner_tags"].feature.names
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}
print(label_names)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


Some training sets are large (Hindi has close to a million sentences), so one epoch can take a while. For a quick first run, set `MAX_TRAIN_SAMPLES` to e.g. `50_000`; set it to `None` to use everything.

In [4]:
MAX_TRAIN_SAMPLES = None

if MAX_TRAIN_SAMPLES is not None:
    dataset["train"] = dataset["train"].shuffle(seed=42).select(range(min(MAX_TRAIN_SAMPLES, len(dataset["train"]))))
print(len(dataset["train"]), "training sentences")

985787 training sentences


## 3. Tokenize and align the labels

The dataset gives one label per **word**, but the tokenizer can split a word into several **tokens**. We pass the words with `is_split_into_words=True` (NandiBERT's tokenizer adds the space before each word itself, as in running text) and use `word_ids()` to map every token back to its word:

- the first token of a word gets the word's label,
- later tokens of the same word, and special tokens such as `<|endoftext|>`, get `-100`, which the loss and the metrics ignore.

In [5]:
MODEL_ID = "FrontiersMind/NandiBERT-150M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

In [6]:
def tokenize_and_align(batch):
    encoded = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=256,
    )

    aligned_labels = []
    for i, word_labels in enumerate(batch["ner_tags"]):
        word_ids = encoded.word_ids(batch_index=i)
        previous_word = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:                # special token
                label_ids.append(-100)
            elif word_id != previous_word:     # first token of a word
                label_ids.append(word_labels[word_id])
            else:                              # rest of the word
                label_ids.append(-100)
            previous_word = word_id
        aligned_labels.append(label_ids)

    encoded["labels"] = aligned_labels
    return encoded

tokenized = dataset.map(
    tokenize_and_align,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

example = tokenized["validation"][0]
for token, label in list(zip(tokenizer.convert_ids_to_tokens(example["input_ids"]), example["labels"]))[:15]:
    print(f"{tokenizer.convert_tokens_to_string([token])!r:20} {id2label.get(label, '-100')}")

Map:   0%|          | 0/985787 [00:00<?, ? examples/s]

Map:   0%|          | 0/13460 [00:00<?, ? examples/s]

Map:   0%|          | 0/864 [00:00<?, ? examples/s]

'<|endoftext|>'      -100
' गुजरात'            B-ORG
' हाई'               I-ORG
'कोर्ट'              -100
' ने'                O
' मंगलवार'           O
' को'                O
' कांग्रेस'          B-ORG
' नेता'              O
' अहमद'              B-PER
' पटेल'              I-PER
' पर'                O
' '                  O
'5'                  -100
' हजार'              O


## 4. Load the token-classification model

`AutoModelForTokenClassification` reuses NandiBERT's pretrained MLM head before a new linear layer, so only that linear layer (7 outputs here) starts from random weights.

We load the weights in float32 and let the Trainer use bf16 mixed precision (`bf16=True` below). Loading the weights directly in bf16 also trains, but the optimizer updates are less precise.

In [7]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
    trust_remote_code=True,
)
print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

[transformers] NandiForTokenClassification LOAD REPORT from: FrontiersMind/NandiBERT-150M
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 146.677511 M


## 5. Metrics and training

NER is scored per **entity**, not per token: an entity counts as correct only if all its words and its type are right. `seqeval` does this from the tag sequences, so we turn the predicted and gold label ids back into tag names, skipping the `-100` positions.

The model card recommends a learning rate of `3e-5` for NER with the MLM head; much lower rates such as `5e-6` work much worse.

In [8]:
def to_tags(predictions, label_ids):
    true_predictions, true_labels = [], []
    for prediction, labels in zip(predictions, label_ids):
        pred_tags, gold_tags = [], []
        for pred_id, label_id in zip(prediction, labels):
            if label_id == -100:
                continue
            pred_tags.append(label_names[pred_id])
            gold_tags.append(label_names[label_id])
        true_predictions.append(pred_tags)
        true_labels.append(gold_tags)
    return true_predictions, true_labels


def compute_metrics(output):
    predictions = output.predictions.argmax(axis=-1)
    true_predictions, true_labels = to_tags(predictions, output.label_ids)
    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

In [ ]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

args = TrainingArguments(
    output_dir=f"./results/naamapadam-{LANG}",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=1,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    bf16=use_bf16,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

/venv/main/lib/python3.12/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


Step,Training Loss,Validation Loss,Precision,Recall,F1
2000,0.179965,0.181077,0.765956,0.772271,0.769100
4000,0.161430,0.174272,0.765869,0.787843,0.776701
6000,0.178769,0.169528,0.768554,0.791082,0.779655
8000,0.165488,0.168405,0.768652,0.799130,0.783595
10000,0.162248,0.166699,0.776407,0.794517,0.785357
12000,0.168002,0.165690,0.777250,0.797363,0.787178
14000,0.163180,0.165023,0.775383,0.797985,0.786522
16000,0.163499,0.164675,0.777696,0.796447,0.786960
18000,0.159277,0.163594,0.776843,0.800504,0.788496
20000,0.170600,0.163146,0.780499,0.798704,0.789497


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 6. Evaluate and save

The test split is only used once, at the end. The per-entity report shows which entity types the model finds hardest.

In [ ]:
print(trainer.evaluate(tokenized["validation"]))

test_output = trainer.predict(tokenized["test"])
print(test_output.metrics)

true_predictions, true_labels = to_tags(test_output.predictions.argmax(axis=-1), test_output.label_ids)
print(classification_report(true_labels, true_predictions, digits=4))

In [ ]:
trainer.save_model(f"./results/naamapadam-{LANG}/final")
tokenizer.save_pretrained(f"./results/naamapadam-{LANG}/final")

## What to try next

- Train on another language by changing `LANG`, or concatenate several languages into one training set.
- Compare learning rates such as `2e-5`, `3e-5` and `5e-5`, and try 2 epochs.